<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-08/AI_Modul_8.5_Praktikum_Pembuatan_Sistem_Prediksi_Sederhana.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 8.5: Praktikum Pembuatan Sistem Prediksi Sederhana
**Mata Kuliah:** AI Academy  
**Institut Pertanian Stiper (INSTIPER) Yogyakarta**  

---

### Capaian Pembelajaran Praktikum:
1. Mengompilasi dan mengekspor model PyTorch terlatih menjadi graf teroptimasi **TorchScript JIT**.
2. Membangun kelas layanan inferensi terpadu (*End-to-End Inference Engine*) dengan filter validasi batas fisik masukan.
3. Mengotomatisasi alur prapemrosesan menggunakan parameter skala terbekukan (*frozen scaler state*) dan pasca-pemrosesan kategori mutu panen.
4. Mengukur efisiensi latensi inferensi dan mensimulasikan sistem prediksi cerdas untuk stasiun sortasi pabrik kelapa sawit.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Konfigurasi lingkungan komputasi
np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Perangkat Komputasi Aktif: {device}")


## 1. Pembangkitan Data Terpadu & Pelatihan Model JST
Kita melatih model `OilPalmYieldMLP` (124 input $\rightarrow$ 64 $\rightarrow$ 32 $\rightarrow$ 1 output) dan membekukan parameter statistika standarisasi $\boldsymbol{\mu}_{train}$ dan $\boldsymbol{\sigma}_{train}$.


In [ ]:
n_samples = 600
n_spectral = 120
afdeling_ids = np.repeat(np.arange(1, 11), n_samples // 10)

wavelengths = np.linspace(400, 1000, n_spectral)
X_spectral = np.zeros((n_samples, n_spectral))
for i in range(n_samples):
    base_sig = 0.15 + 0.35 * np.exp(-((wavelengths - 550)**2) / (2 * 45**2)) +                       0.50 * (1.0 / (1.0 + np.exp(-(wavelengths - 700) / 20)))
    noise = np.random.normal(0, 0.02, n_spectral)
    X_spectral[i] = base_sig * np.random.uniform(0.85, 1.15) + noise

pH = np.random.normal(5.2, 0.4, (n_samples, 1))
nitrogen = np.random.normal(120.0, 25.0, (n_samples, 1))
moisture = np.random.uniform(30.0, 70.0, (n_samples, 1))
rainfall = np.random.uniform(100.0, 350.0, (n_samples, 1))
X_sensor = np.hstack([pH, nitrogen, moisture, rainfall])
X_raw = np.hstack([X_spectral, X_sensor])

y_raw = 19.5 + 4.5 * X_spectral[:, 65] - 0.015 * np.abs(pH[:, 0] - 5.5)**2 +         0.012 * (nitrogen[:, 0] / 10.0) + np.random.normal(0, 0.35, n_samples)
y_raw = y_raw.reshape(-1, 1)

train_mask = np.isin(afdeling_ids, [1, 2, 3, 4, 5, 6])
test_mask  = np.isin(afdeling_ids, [9, 10])

# Parameter Normalisasi Terbekukan
mu_train = np.mean(X_raw[train_mask], axis=0, keepdims=True)
sigma_train = np.std(X_raw[train_mask], axis=0, keepdims=True) + 1e-8

X_train_norm = (X_raw[train_mask] - mu_train) / sigma_train
y_train = y_raw[train_mask]

class AgroDataset(Dataset):
    def __init__(self, X_data, y_data):
        self.X = torch.tensor(X_data, dtype=torch.float32)
        self.y = torch.tensor(y_data, dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(AgroDataset(X_train_norm, y_train), batch_size=32, shuffle=True)

class OilPalmYieldMLP(nn.Module):
    def __init__(self, in_features=124, hidden1=64, hidden2=32, out_features=1):
        super(OilPalmYieldMLP, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(p=0.15),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(p=0.10),
            nn.Linear(hidden2, out_features)
        )
    def forward(self, x):
        return self.net(x)

model = OilPalmYieldMLP().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.005, weight_decay=1e-4)
criterion = nn.MSELoss()

model.train()
for epoch in range(25):
    for bx, by in train_loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        loss = criterion(model(bx), by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

# Simpan bobot PyTorch asli
torch.save(model.state_dict(), 'oilpalm_trained_model.pt')
print("Model PyTorch berhasil dilatih dan disimpan ke: oilpalm_trained_model.pt")


## 2. Kompilasi Graf Portabel: TorchScript JIT Export
Kita mengompilasi model ke format **TorchScript** menggunakan metode *tracing* (`torch.jit.trace`) sehingga model dapat dieksekusi secara mandiri tanpa dependensi kode Python arsitektur asli.


In [ ]:
model.eval()
model_cpu = model.cpu() # Ekspor ke CPU agar kompatibel dengan komputer pabrik

dummy_input = torch.randn(1, 124, dtype=torch.float32)
traced_script_module = torch.jit.trace(model_cpu, dummy_input)

jit_filename = 'oilpalm_predictor_jit.pt'
traced_script_module.save(jit_filename)

print("=================================================================")
print(f"[OK] Model berhasil diekspor ke TorchScript: {jit_filename}")
print("Verifikasi: Memuat kembali model TorchScript dari disk...")
loaded_jit = torch.jit.load(jit_filename)
loaded_jit.eval()

# Uji Keselarasan Numerik
with torch.no_grad():
    out_orig = model_cpu(dummy_input)
    out_jit  = loaded_jit(dummy_input)
    diff = torch.abs(out_orig - out_jit).item()

print(f"Prediksi Model Asli PyTorch   : {out_orig.item():.6f}")
print(f"Prediksi Model TorchScript JIT: {out_jit.item():.6f}")
print(f"Selisih Absolut Relatif       : {diff:.8f}")
print("=================================================================")
assert diff < 1e-5, "Model TorchScript identik sempurna dengan model asli!"
print("[OK] Verifikasi TorchScript Berhasil!")


## 3. Konstruksi Kelas Mesin Inferensi Produksi (*Inference Engine*)
Kita membungkus seluruh tahapan operasional ke dalam kelas `OilPalmInferencePipeline`:
1. **Validasi Masukan:** Memeriksa nilai kosong (NaN/Inf) dan rentang fisik batas wajar.
2. **Prapemrosesan Otomatis:** Mengaplikasikan standarisasi dengan scaler terbekukan.
3. **Eksekusi Cepat TorchScript:** Menghitung prediksi rendemen tanpa gradien.
4. **Pasca-Pemrosesan Keputusan:** Menetapkan kategori mutu panen (Grade A, B, C) dan instruksi pabrik.


In [ ]:
class OilPalmInferencePipeline:
    """Mesin inferensi mandiri siap produksi untuk stasiun sortasi kelapa sawit."""
    def __init__(self, jit_path, mu_train, sigma_train):
        self.model = torch.jit.load(jit_path)
        self.model.eval()
        self.mu = np.array(mu_train, dtype=np.float32).reshape(1, -1)
        self.sigma = np.array(sigma_train, dtype=np.float32).reshape(1, -1)
        self.sigma[self.sigma == 0] = 1.0

    def validate_inputs(self, raw_input):
        arr = np.array(raw_input, dtype=np.float32).flatten()
        if len(arr) != 124:
            raise ValueError(f"[GALAT VALIDASI] Panjang fitur tidak valid: {len(arr)} (Diharapkan 124 fitur)")
        if np.isnan(arr).any() or np.isinf(arr).any():
            raise ValueError("[GALAT VALIDASI] Terdapat nilai kosong (NaN) atau tak terhingga (Inf)")
            
        # Pengecekan Batas Fisik Sensor Tanah (indeks 120 s.d. 123)
        ph_val = arr[120]
        n_val  = arr[121]
        if ph_val < 3.0 or ph_val > 9.0:
            raise ValueError(f"[GALAT FISIK] Nilai pH tanah ({ph_val:.1f}) di luar batas fisik wajar (3.0 - 9.0)")
        if n_val < 0.0 or n_val > 500.0:
            raise ValueError(f"[GALAT FISIK] Nilai Nitrogen tanah ({n_val:.1f} ppm) di luar batas wajar (0 - 500 ppm)")
            
        # Pengecekan Reflektansi Spektral
        if np.min(arr[:120]) < -0.1 or np.max(arr[:120]) > 2.0:
            raise ValueError("[GALAT FISIK] Nilai reflektansi spektral optik di luar jangkauan sensor")
            
        return arr.reshape(1, -1)

    def predict(self, raw_features):
        try:
            # 1. Validasi Masukan
            clean_input = self.validate_inputs(raw_features)
            
            # 2. Prapemrosesan Otomatis
            norm_input = (clean_input - self.mu) / (self.sigma + 1e-8)
            tensor_input = torch.tensor(norm_input, dtype=torch.float32)
            
            # 3. Inferensi TorchScript
            t_start = time.perf_counter()
            with torch.no_grad():
                pred_tensor = self.model(tensor_input)
            t_end = time.perf_counter()
            
            y_pred = float(pred_tensor.item())
            lat_ms = (t_end - t_start) * 1000.0
            
            # 4. Pasca-Pemrosesan Keputusan Mutu
            if y_pred >= 23.0:
                grade = "Grade A (Super)"
                rekomendasi = "Prioritas ekstraksi utama, harga beli premium."
            elif y_pred >= 21.0:
                grade = "Grade B (Standar Pabrik)"
                rekomendasi = "Ekstraksi reguler sesuai indeks harga standar."
            else:
                grade = "Grade C (Sub-Standar)"
                rekomendasi = "Sortasi ulang, dikenakan penalti potongan harga."
                
            return {
                'status': 'SUCCESS',
                'rendemen_cpo_pct': round(y_pred, 2),
                'grade_mutu': grade,
                'rekomendasi_pabrik': rekomendasi,
                'latensi_ms': round(lat_ms, 3)
            }
        except Exception as e:
            return {
                'status': 'ERROR',
                'pesan_kesalahan': str(e)
            }

# Inisialisasi Mesin Inferensi
pipeline = OilPalmInferencePipeline(jit_filename, mu_train, sigma_train)
print("[OK] Mesin Inferensi Produksi Siap Melayani Prediksi!")


## 4. Pengujian Sistem pada Berbagai Skenario Lapangan
Kita menguji ketangguhan sistem prediksi pada 3 skenario riil:
1. **Skenario 1:** Sampel Pohon Unggul (Pita red-edge kuat, pH 5.4, N 130 ppm).
2. **Skenario 2:** Sampel Pohon Kerdil/Rendah (Pita red-edge lemah, pH 4.2, N 75 ppm).
3. **Skenario 3:** Sampel Cacat Sensorik (pH 12.5 atau data rusak) untuk menguji proteksi sistem.


In [ ]:
# Skenario 1: Sampel Normal Rendemen Tinggi
sample_normal_high = X_raw[0].copy()
sample_normal_high[65] = 0.95 # Puncak red-edge tinggi
sample_normal_high[120] = 5.5 # pH optimal
sample_normal_high[121] = 135.0 # Nitrogen baik

# Skenario 2: Sampel Normal Rendemen Rendah
sample_normal_low = X_raw[0].copy()
sample_normal_low[65] = 0.45 # Puncak red-edge rendah
sample_normal_low[120] = 4.1 # pH masam ekstrem
sample_normal_low[121] = 70.0 # Kurang hara

# Skenario 3: Sampel Cacat Sensorik (Nilai pH tidak realistis)
sample_corrupt = X_raw[0].copy()
sample_corrupt[120] = 13.8 # pH basa ekstrem mustahil

print("=================================================================")
print("                   UJI SKENARIO LAPANGAN                         ")
print("=================================================================")

res1 = pipeline.predict(sample_normal_high)
print(f"Skenario 1 (Pohon Unggul) : Prediksi = {res1['rendemen_cpo_pct']}% | {res1['grade_mutu']} | Latensi: {res1['latensi_ms']} ms")

res2 = pipeline.predict(sample_normal_low)
print(f"Skenario 2 (Pohon Rendah) : Prediksi = {res2['rendemen_cpo_pct']}% | {res2['grade_mutu']} | Latensi: {res2['latensi_ms']} ms")

res3 = pipeline.predict(sample_corrupt)
print(f"Skenario 3 (Sensor Rusak) : Status = {res3['status']} | Pesan = {res3['pesan_kesalahan']}")
print("=================================================================")


## 5. Benchmark Latensi Komputasi Inferensi (100 Iterasi)
Kita menguji perbandingan kecepatan eksekusi antara PyTorch Eager Mode dengan TorchScript JIT Engine.


In [ ]:
dummy_tensor = torch.randn(1, 124, dtype=torch.float32)
n_iter = 100

# Benchmark PyTorch Eager (Python Overhead)
model_cpu.eval()
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_iter):
        _ = model_cpu(dummy_tensor)
t_eager = (time.perf_counter() - t0) / n_iter * 1000.0

# Benchmark TorchScript JIT
t0 = time.perf_counter()
with torch.no_grad():
    for _ in range(n_iter):
        _ = loaded_jit(dummy_tensor)
t_jit = (time.perf_counter() - t0) / n_iter * 1000.0

print("=================================================================")
print("           BENCHMARK KECEPATAN INFERENSI (RATA-RATA)             ")
print("=================================================================")
print(f"PyTorch Eager Mode (Python) : {t_eager:.3f} milidetik per prediksi")
print(f"TorchScript JIT Mode (C++)  : {t_jit:.3f} milidetik per prediksi")
speedup = t_eager / t_jit if t_jit > 0 else 1.0
print(f"Faktor Percepatan Komputasi : {speedup:.2f}x Lebih Cepat!")
print("=================================================================")

# Visualisasi Bar
plt.figure(figsize=(6, 4))
plt.bar(['PyTorch Eager', 'TorchScript JIT'], [t_eager, t_jit], color=['#E53E3E', '#3182CE'], width=0.4, edgecolor='k')
plt.ylabel('Latensi Inferensi (ms) - Lebih Rendah Lebih Cepat')
plt.title('Perbandingan Latensi Komputasi Inferensi', fontsize=11, fontweight='bold')
for i, v in enumerate([t_eager, t_jit]):
    plt.text(i, v + 0.05, f"{v:.2f} ms", ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print("[OK] Seluruh Rangkaian Siklus Hidup Deep Learning (Modul 8.1 - 8.11) PARIPURNA!")
